# Phase 3 — Rhetorical Role Prediction (CL + IT) · Kaggle 2× T4

**Inputs:** `prefixlm-run1` (tokenizer) + `phase1-models` (your four Phase-1 models).
Optional: your Phase-2 notebook output / `phase2-weights` if you want to reuse `ner_model.py` code.
Settings: GPU T4 ×2, Internet On, secret `HF_TOKEN` (tick it in Add-ons → Secrets).

**Yours (rule 17.5):** `rr_model.py` (sentence representation, head, adaptation methods incl. LoRMA, KD student) and every `TODO`.
**Harness:** data, sentence windows (sentence-only or with context), training loop, dev macro-F1 per domain + combined, KD plumbing, prediction.json for both domains + zip.

Order: cells 1-6 once (tests pass, data check, SMOKE TEST PASSED) → write `rr_model.py` → model check → experiments.

In [ ]:
# 1 — setup
import os, json, glob, shutil, time, subprocess, sys
os.chdir("/kaggle/working")
import torch
NGPU = torch.cuda.device_count(); print("GPUs:", NGPU)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not available:", e)

def find(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

In [ ]:
# 2 — Phase-3 data + tokenizer
from huggingface_hub import hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
P3 = {}
for f in ["Train/CL_train.json", "Train/CL_dev.json", "Train/IT_train.json", "Train/IT_dev.json",
          "Train/label_vocab.json", "Test/CL_test.json", "Test/IT_test.json"]:
    P3[f.split("/")[-1]] = hf_hub_download(REPO, f"Phase_3_RR/{f}", repo_type="dataset")
print(json.dumps(P3, indent=1))
os.makedirs("tokenizer", exist_ok=True)
for fn in ["vocab.json", "merges.txt"]:
    shutil.copy(find(fn)[0], f"tokenizer/{fn}")
print("label_vocab.json:", open(P3["label_vocab.json"]).read())

In [ ]:
%%writefile rr_data.py
"""rr_data.py — Phase-3 Rhetorical Role data handling."""
import json
from collections import Counter


# ------------------------------------------------------------------ loading
def read_label_vocab(path):
    obj = json.load(open(path, encoding="utf-8"))
    if isinstance(obj, dict) and "labels" in obj:
        obj = obj["labels"]
    if isinstance(obj, dict):
        items = list(obj.items())
        if all(isinstance(v, int) for _, v in items):
            return [k for k, _ in sorted(items, key=lambda kv: kv[1])]
        return [v for _, v in sorted(items, key=lambda kv: int(kv[0]))]
    return list(obj)


def load_docs(path):
    obj = json.load(open(path, encoding="utf-8"))
    docs = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, list):
                docs[str(k)] = {"text": list(v), "labels": None}
            else:
                docs[str(k)] = {"text": list(v.get("text", v.get("sentences"))),
                                "labels": list(v["labels"]) if v.get("labels") is not None else None}
    else:
        for r in obj:
            did = str(r.get("id", r.get("doc_id")))
            docs[did] = {"text": list(r.get("text", r.get("sentences"))),
                         "labels": list(r["labels"]) if r.get("labels") is not None else None}
    for d, v in docs.items():
        if v["labels"] is not None:
            assert len(v["labels"]) == len(v["text"]), f"{d}: {len(v['text'])} sentences vs {len(v['labels'])} labels"
    return docs


# ------------------------------------------------------------------ tokenization + windows
def tokenize_sentences(docs, encode_ids, prefix_space=True, max_sent_tokens=None):
    out = {}
    for d, v in docs.items():
        sents = []
        for s in v["text"]:
            s = s if isinstance(s, str) else str(s)
            ids = encode_ids((" " + s) if (prefix_space and s and not s[0].isspace()) else s) if s.strip() else []
            if max_sent_tokens:
                ids = ids[:max_sent_tokens]
            sents.append(ids)
        out[d] = sents
    return out


def _encode_window(sent_ids_list, order, sep_id):
    ids, slots = [], []
    for k, si in enumerate(order):
        if k and sep_id is not None:
            ids.append(sep_id); slots.append(-1)
        toks = sent_ids_list[si]
        ids.extend(toks); slots.extend([k] * len(toks))
    return ids, slots


def build_windows(tok_docs, mode, window, sep_id=None, target_frac=0.5):
    out = []
    sep = 1 if sep_id is not None else 0
    for d, sents in tok_docs.items():
        n = len(sents)
        L = [max(1, len(s)) for s in sents]
        if mode == "sentence":
            for i in range(n):
                s = sents[i][:window] or [sep_id if sep_id is not None else 0]
                out.append(dict(doc=d, ids=list(s), slots=[0] * len(s), sent_idx=[i], target=[True]))
            continue
        assert mode == "context", mode
        budget_t = max(1, int(window * target_frac))
        i = 0
        while i < n:
            j, used = i, 0
            while j < n and (j == i or used + sep + L[j] <= budget_t):
                used += (sep if j > i else 0) + L[j]; j += 1
            tgt = list(range(i, j))
            used = min(used, window)
            lo, hi = i - 1, j
            left, right = [], []
            turn = 0
            while lo >= 0 or hi < n:
                take_left = (turn % 2 == 0 and lo >= 0) or hi >= n
                k = lo if take_left else hi
                cost = sep + L[k]
                if used + cost > window:
                    break
                used += cost
                if take_left:
                    left.insert(0, k); lo -= 1
                else:
                    right.append(k); hi += 1
                turn += 1
            order = left + tgt + right
            seqs = [sents[k] if sents[k] else [sep_id if sep_id is not None else 0] for k in range(n)]
            ids, slots = _encode_window(seqs, order, sep_id)
            if len(ids) > window:
                ids, slots = ids[:window], slots[:window]
            present = sorted(set(s for s in slots if s >= 0))
            keep = {k: r for r, k in enumerate(present)}
            slots = [keep[s] if s >= 0 else -1 for s in slots]
            sidx = [order[k] for k in present]
            tset = set(tgt)
            out.append(dict(doc=d, ids=ids, slots=slots, sent_idx=sidx, target=[s in tset for s in sidx]))
            i = j
    return out


def coverage_check(windows, tok_docs):
    seen = Counter()
    for w in windows:
        for si, t in zip(w["sent_idx"], w["target"]):
            if t:
                seen[(w["doc"], si)] += 1
    want = {(d, i) for d, s in tok_docs.items() for i in range(len(s))}
    missing = want - set(seen)
    dup = [k for k, c in seen.items() if c > 1]
    return missing, dup


# ------------------------------------------------------------------ scoring
def macro_scores(pred, gold, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    n = correct = 0
    for d, g in gold.items():
        p = pred[d]
        assert len(p) == len(g), f"{d}: {len(p)} predictions for {len(g)} sentences"
        for a, b in zip(p, g):
            n += 1
            if a == b:
                tp[a] += 1; correct += 1
            else:
                fp[a] += 1; fn[b] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = dict(p=P, r=R, f1=2 * P * R / (P + R) if P + R else 0.0, support=tp[l] + fn[l], predicted=tp[l] + fp[l])
    present = [l for l in labels if per[l]["support"] or per[l]["predicted"]]
    m = lambda key, ls: sum(per[l][key] for l in ls) / max(1, len(ls))
    return dict(macro_f1=m("f1", labels), macro_p=m("p", labels), macro_r=m("r", labels),
                macro_f1_present=m("f1", present), accuracy=correct / max(1, n), per_label=per, n=n)


def format_scores(sc, title=""):
    lines = [f"{title}macro-F1 {sc['macro_f1']:.4f} (labels present: {sc['macro_f1_present']:.4f}) | "
             f"macro-P {sc['macro_p']:.4f} | macro-R {sc['macro_r']:.4f} | acc {sc['accuracy']:.4f} | n {sc['n']}"]
    for l, v in sc["per_label"].items():
        lines.append(f"  {l:24s} F1 {v['f1']:.3f}  P {v['p']:.3f}  R {v['r']:.3f}  (support {v['support']}, predicted {v['predicted']})")
    return "\n".join(lines)


def validate_prediction(pred, test_docs, labels):
    missing = set(test_docs) - set(pred)
    extra = set(pred) - set(test_docs)
    assert not missing, f"{len(missing)} test docs missing, e.g. {list(missing)[:3]}"
    assert not extra, f"{len(extra)} unknown doc ids, e.g. {list(extra)[:3]}"
    lab = set(labels)
    n = 0
    for d, p in pred.items():
        assert isinstance(p, list) and len(p) == len(test_docs[d]["text"]), \
            f"{d}: {len(p) if isinstance(p, list) else type(p)} predictions for {len(test_docs[d]['text'])} sentences"
        bad = [x for x in p if x not in lab]
        assert not bad, f"{d}: labels not in vocab: {bad[:3]}"
        n += len(p)
    return n


def dataset_stats(docs, tok_docs, labels):
    nsent = [len(v["text"]) for v in docs.values()]
    stoks = sorted(len(s) for d in tok_docs.values() for s in d)
    dtoks = sorted(sum(len(s) for s in d) for d in tok_docs.values())
    cnt = Counter(l for v in docs.values() if v["labels"] for l in v["labels"])
    q = lambda a, f: a[min(len(a) - 1, int(len(a) * f))] if a else 0
    return dict(docs=len(docs), sentences=sum(nsent), sents_per_doc_p50=sorted(nsent)[len(nsent) // 2] if nsent else 0,
                sent_tokens_p50=q(stoks, .5), sent_tokens_p95=q(stoks, .95), sent_tokens_max=stoks[-1] if stoks else 0,
                doc_tokens_p50=q(dtoks, .5), doc_tokens_max=dtoks[-1] if dtoks else 0,
                labels=dict(cnt.most_common()), unknown_labels=sorted(set(cnt) - set(labels)))

In [ ]:
%%writefile test_rr_data.py
"""Synthetic tests for rr_data.py (python test_rr_data.py)."""
import json, os, random, tempfile

import rr_data as rd

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name} {detail}")


random.seed(0)
LAB = ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent", "PrecedentReliedUpon",
       "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision",
       "RulingByPresentCourt", "None"]
enc = lambda s: [hash(w) % 30000 + 5 for w in s.split()]

docs = {}
for d in range(30):
    n = random.randint(1, 80)
    sents = [" ".join(f"w{random.randint(0, 999)}" for _ in range(random.choice([0, 1, 3, 8, 20, 60, 400]))) for _ in range(n)]
    docs[f"doc{d}"] = {"text": sents, "labels": [random.choice(LAB) for _ in range(n)]}
tok = rd.tokenize_sentences(docs, enc, prefix_space=True)

# 1-3 windows: coverage, length, order, slot consistency
for mode in ("sentence", "context"):
    for W, sep in [(64, None), (256, 1), (1024, 1), (30, 1)]:
        wins = rd.build_windows(tok, mode, W, sep_id=sep, target_frac=0.5)
        missing, dup = rd.coverage_check(wins, tok)
        good_len = all(len(w["ids"]) <= W and len(w["ids"]) == len(w["slots"]) for w in wins)
        good_slots = all(max(w["slots"]) == len(w["sent_idx"]) - 1 and min(s for s in w["slots"] if s >= 0) == 0 for w in wins)
        ordered = all(w["sent_idx"] == sorted(w["sent_idx"]) and
                      w["sent_idx"] == list(range(w["sent_idx"][0], w["sent_idx"][0] + len(w["sent_idx"]))) for w in wins)
        tgt_contig = all((lambda t: t == list(range(t[0], t[0] + len(t))))([s for s, f in zip(w["sent_idx"], w["target"]) if f])
                         for w in wins)
        check(f"1 {mode} W={W} sep={sep}: every sentence scored exactly once", not missing and not dup,
              f"(missing {len(missing)}, dup {len(dup)}, windows {len(wins)})")
        check(f"2 {mode} W={W}: lengths <= window, slots consistent", good_len and good_slots)
        check(f"3 {mode} W={W}: sentence order kept, targets contiguous", ordered and tgt_contig)


wins = rd.build_windows(tok, "context", 1024, sep_id=1, target_frac=0.3)
check("4 context windows contain non-target sentences", any(not all(w["target"]) for w in wins))


gold = {"a": ["Fact", "Fact", "Issue"], "b": ["None"]}
pred = {"a": ["Fact", "Issue", "Issue"], "b": ["None"]}
sc = rd.macro_scores(pred, gold, LAB)
check("5 scorer", abs(sc["per_label"]["Fact"]["f1"] - 2 / 3) < 1e-9 and sc["per_label"]["Issue"]["f1"] == 2 / 3
      and sc["accuracy"] == 0.75 and abs(sc["macro_f1_present"] - (2 / 3 + 2 / 3 + 1) / 3) < 1e-9)


tmp = tempfile.mkdtemp()
json.dump([{"id": k, "text": v["text"], "labels": v["labels"]} for k, v in docs.items()], open(f"{tmp}/a.json", "w"))
json.dump({k: {"text": v["text"], "labels": v["labels"]} for k, v in docs.items()}, open(f"{tmp}/b.json", "w"))
json.dump({k: v["text"] for k, v in docs.items()}, open(f"{tmp}/c.json", "w"))
a, b, c = rd.load_docs(f"{tmp}/a.json"), rd.load_docs(f"{tmp}/b.json"), rd.load_docs(f"{tmp}/c.json")
check("6 loaders agree", a == b and all(c[k]["text"] == a[k]["text"] and c[k]["labels"] is None for k in a))
json.dump(LAB, open(f"{tmp}/v1.json", "w")); json.dump({l: i for i, l in enumerate(LAB)}, open(f"{tmp}/v2.json", "w"))
json.dump({"labels": LAB}, open(f"{tmp}/v3.json", "w"))
check("6 label vocab layouts", rd.read_label_vocab(f"{tmp}/v1.json") == rd.read_label_vocab(f"{tmp}/v2.json")
      == rd.read_label_vocab(f"{tmp}/v3.json") == LAB)


p = {k: v["labels"] for k, v in docs.items()}
check("7 validator accepts", rd.validate_prediction(p, docs, LAB) == sum(len(v["text"]) for v in docs.values()))
bad = dict(p); bad["doc0"] = bad["doc0"][:-1] if len(bad["doc0"]) > 1 else ["Fact", "Fact"]
try:
    rd.validate_prediction(bad, docs, LAB); check("7 validator rejects wrong count", False)
except AssertionError:
    check("7 validator rejects wrong count", True)

print("\nALL TESTS PASSED" if ok else "\nSOME TESTS FAILED")

In [ ]:
%%writefile rr_train.py
"""rr_train.py — Phase-3 Rhetorical Role fine-tuning harness."""
import importlib, json, math, os, random, sys, time, zipfile

import numpy as np
import torch
import torch.nn.functional as F

import rr_data as rd

CFG = json.load(open(sys.argv[1]))
D, TR, MODEL_CFG = CFG["data"], CFG["train"], CFG["model"]
KD = CFG.get("kd") or {}
OUT, NAME = CFG["out_dir"], CFG["run_name"]
os.makedirs(OUT, exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = int(TR.get("seed", 0))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


def log(*a):
    print(*a, flush=True)


# ------------------------------------------------------------------ data
from tokenizers import ByteLevelBPETokenizer
TOK = ByteLevelBPETokenizer(vocab=os.path.join(D["tokenizer_dir"], "vocab.json"),
                            merges=os.path.join(D["tokenizer_dir"], "merges.txt"))
PAD_ID = TOK.token_to_id("<pad>") or 0
SEP_ID = TOK.token_to_id(D["sep"]) if D.get("sep") else None
if D.get("sep"):
    assert SEP_ID is not None, f"separator {D['sep']!r} is not a token in the vocabulary"
LABELS = rd.read_label_vocab(D["label_vocab"])
L2I = {l: i for i, l in enumerate(LABELS)}
C = len(LABELS)
W = int(D["window"])
MODE = D["mode"]


def encode_ids(s):
    return TOK.encode(s).ids


DOMS = list(D["domains"])
SPLITS = {}
for dom in DOMS:
    for split, path in D["domains"][dom].items():
        for did, rec in rd.load_docs(path).items():
            SPLITS.setdefault(split, {})[f"{dom}::{did}"] = dict(rec, dom=dom, id=did)
train_splits = D.get("train_splits", ["train"])
train_doms = D.get("train_domains", DOMS)
TRAIN = {k: v for s in train_splits for k, v in SPLITS.get(s, {}).items() if v["dom"] in train_doms}
DEVS = {k: v for k, v in SPLITS.get("dev", {}).items()} if "dev" not in train_splits else {}
TEST = SPLITS.get("test", {})
for name, part in (("train", TRAIN), ("dev", DEVS)):
    bad = {l for v in part.values() for l in (v["labels"] or []) if l not in L2I}
    assert not bad, f"{name} has labels not in label_vocab: {sorted(bad)[:5]}"
log(f"labels {C} | train docs {len(TRAIN)} ({'+'.join(train_splits)}; {','.join(train_doms)}) | dev docs {len(DEVS)} | "
    f"test docs {len(TEST)} | mode {MODE} window {W} sep {D.get('sep')}")

TOKS = {}


def toks(part):
    need = [k for k in part if k not in TOKS]
    if need:
        TOKS.update(rd.tokenize_sentences({k: part[k] for k in need}, encode_ids, D.get("prefix_space", True),
                                          D.get("max_sent_tokens")))
    return {k: TOKS[k] for k in part}


def windows(part):
    ws = rd.build_windows(toks(part), MODE, W, sep_id=SEP_ID, target_frac=float(D.get("target_frac", 0.5)))
    missing, dup = rd.coverage_check(ws, toks(part))
    assert not missing and not dup, f"window coverage broken: missing {len(missing)} dup {len(dup)}"
    return ws


TEACH = None
if KD.get("teacher_logits"):
    z = np.load(KD["teacher_logits"])
    meta = json.loads(str(z["__meta__"]))
    assert meta["labels"] == LABELS, "teacher used a different label order"
    TEACH = {k[2:]: z[k] for k in z.files if k.startswith("d:")}
    log(f"KD: teacher logits for {len(TEACH)} docs | T={KD['temperature']} alpha={KD['alpha']}")


def collate(batch, part, with_labels, with_teacher=False):
    B = len(batch)
    T = max(len(w["ids"]) for w in batch)
    S = max(len(w["sent_idx"]) for w in batch)
    x = torch.full((B, T), PAD_ID, dtype=torch.long)
    m = torch.zeros((B, T), dtype=torch.bool)
    sid = torch.full((B, T), -1, dtype=torch.long)
    y = torch.full((B, S), -100, dtype=torch.long)
    tch = torch.zeros((B, S, C), dtype=torch.float32) if (with_teacher and TEACH is not None) else None
    for i, w in enumerate(batch):
        n = len(w["ids"])
        x[i, :n] = torch.tensor(w["ids"]); m[i, :n] = True
        sid[i, :n] = torch.tensor(w["slots"])
        rec = part[w["doc"]]
        for k, (si, t) in enumerate(zip(w["sent_idx"], w["target"])):
            if t and with_labels:
                y[i, k] = L2I[rec["labels"][si]]
            if t and tch is not None:
                tch[i, k] = torch.from_numpy(TEACH[w["doc"]][si].astype(np.float32))
    return x, m, sid, S, y, tch


# ------------------------------------------------------------------ model
model_lib = importlib.import_module(CFG.get("model_module", "rr_model"))
model = model_lib.build_rr_model(dict(MODEL_CFG), C).to(DEV)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"parameters: total {n_total/1e6:.2f}M | trainable {n_train/1e6:.3f}M ({100*n_train/max(1,n_total):.2f}%)")
assert n_train > 0, "no trainable parameters"

if CFG.get("load_trained"):
    sd = torch.load(CFG["load_trained"], map_location="cpu", weights_only=False)["trained"]
    cur = dict(model.named_parameters())
    miss = [k for k in sd if k not in cur]
    assert not miss, f"load_trained: names not in this model, e.g. {miss[:3]}"
    with torch.no_grad():
        for k, v in sd.items():
            cur[k].copy_(v.to(cur[k].device, cur[k].dtype))
    log(f"loaded {len(sd)} trained tensors from {CFG['load_trained']}")


@torch.no_grad()
def predict_logits(part):
    model.eval()
    ws = sorted(windows(part), key=lambda w: len(w["ids"]))
    out = {k: np.zeros((len(v["text"]), C), np.float32) for k, v in part.items()}
    bs = int(TR.get("eval_batch_size", TR["batch_size"]))
    for i in range(0, len(ws), bs):
        b = ws[i:i + bs]
        x, m, sid, S, _, _ = collate(b, part, with_labels=False)
        with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
            lg = model(x.to(DEV), m.to(DEV), sid.to(DEV), S).float().cpu().numpy()
        assert lg.shape[:2] == (len(b), S), f"model returned {lg.shape}, expected [{len(b)}, {S}, {C}]"
        for j, w in enumerate(b):
            for k, (si, t) in enumerate(zip(w["sent_idx"], w["target"])):
                if t:
                    out[w["doc"]][si] = lg[j, k]
    model.train()
    return out


def labels_from(lg):
    return {k: [LABELS[i] for i in v.argmax(-1)] for k, v in lg.items()}


def dev_scores(lg):
    pred = labels_from(lg)
    res = {}
    for dom in DOMS:
        ks = [k for k in DEVS if DEVS[k]["dom"] == dom]
        if ks:
            res[dom] = rd.macro_scores({k: pred[k] for k in ks}, {k: DEVS[k]["labels"] for k in ks}, LABELS)
    res["combined"] = rd.macro_scores(pred, {k: v["labels"] for k, v in DEVS.items()}, LABELS)
    return res, pred


# ------------------------------------------------------------------ training
train_w = windows(TRAIN)
bs, acc = int(TR["batch_size"]), int(TR.get("grad_accum", 1))
epochs = int(TR["epochs"])
total_steps = math.ceil(len(train_w) / (bs * acc)) * epochs
warm = int(float(TR.get("warmup_frac", 0)) * total_steps)
base_lr = float(TR["lr"])
groups = model.param_groups(base_lr) if hasattr(model, "param_groups") else \
    [{"params": [p for p in model.parameters() if p.requires_grad]}]
opt = torch.optim.AdamW(groups, lr=base_lr, weight_decay=float(TR.get("weight_decay", 0.0)))
for g in opt.param_groups:
    g.setdefault("initial_lr", g["lr"])
scaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
cw = TR.get("class_weights")
if isinstance(cw, dict):
    cw = [float(cw.get(l, 1.0)) for l in LABELS]
cw = torch.tensor(cw, dtype=torch.float32, device=DEV) if cw is not None else None
SELECT = CFG.get("select", "combined")
log(f"train windows {len(train_w)} | steps/epoch {math.ceil(len(train_w) / (bs * acc))} | select best by dev {SELECT}")


def lr_mult(step):
    if step < warm:
        return (step + 1) / max(1, warm)
    t = (step - warm) / max(1, total_steps - warm)
    sch = TR.get("schedule", "linear")
    return 1.0 if sch == "constant" else max(0.0, 1 - t) if sch == "linear" else 0.5 * (1 + math.cos(math.pi * t))


best, best_state, history = -1.0, None, []
if epochs == 0:
    best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
step, t0 = 0, time.time()
max_min = float(TR.get("max_minutes", 1e9))
if DEV.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
model.train()
for ep in range(1, epochs + 1):
    random.shuffle(train_w)
    run_loss, nb = 0.0, 0
    for i in range(0, len(train_w), bs):
        x, m, sid, S, y, tch = collate(train_w[i:i + bs], TRAIN, with_labels=True, with_teacher=True)
        x, m, sid, y = x.to(DEV), m.to(DEV), sid.to(DEV), y.to(DEV)
        with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
            lg = model(x, m, sid, S)
        lg = lg.float()
        loss = F.cross_entropy(lg.reshape(-1, C), y.reshape(-1), weight=cw, ignore_index=-100)
        if tch is not None:
            Tk, a = float(KD["temperature"]), float(KD["alpha"])
            v = y != -100
            kl = F.kl_div(F.log_softmax(lg[v] / Tk, -1), F.softmax(tch.to(DEV)[v] / Tk, -1), reduction="batchmean")
            loss = (1 - a) * loss + a * Tk * Tk * kl
        scaler.scale(loss / acc).backward()
        nb += 1; run_loss += loss.item()
        if nb % acc == 0:
            for g in opt.param_groups:
                g["lr"] = g["initial_lr"] * lr_mult(step)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], float(TR.get("grad_clip", 1.0)))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        if (time.time() - t0) / 60 > max_min:
            break
    rec = dict(epoch=ep, train_loss=run_loss / max(1, nb), minutes=(time.time() - t0) / 60)
    if DEVS:
        sc, _ = dev_scores(predict_logits(DEVS))
        for k, v in sc.items():
            rec[f"dev_{k}_f1"] = v["macro_f1"]
        score = sc[SELECT]["macro_f1"]
    else:
        score = ep
    history.append(rec)
    log(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
    if score > best:
        best = score
        best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
        log(f"  best so far (epoch {ep})")
    if (time.time() - t0) / 60 > max_min:
        log(f"time limit {max_min} min reached"); break
train_min = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEV.type == "cuda" else 0.0

cur = dict(model.named_parameters())
with torch.no_grad():
    for k, v in best_state.items():
        cur[k].copy_(v.to(DEV))
torch.save(dict(trained=best_state, cfg=CFG, labels=LABELS), os.path.join(OUT, f"{NAME}_trained_params.pt"))

result = dict(run=NAME, labels=LABELS, mode=MODE, window=W, params_total=n_total, params_trainable=n_train,
              train_minutes=train_min, peak_mem_gb=peak_gb, history=history)
if DEVS:
    t1 = time.time()
    dlg = predict_logits(DEVS)
    sc, dpred = dev_scores(dlg)
    result.update(dev={k: v for k, v in sc.items()}, dev_infer_seconds=time.time() - t1)
    for k in [d for d in DOMS if d in sc] + ["combined"]:
        log(rd.format_scores(sc[k], f"BEST on dev [{k}] "))
    json.dump({f"{DEVS[k]['dom']}::{DEVS[k]['id']}": v for k, v in dpred.items()},
              open(os.path.join(OUT, f"{NAME}_dev_prediction.json"), "w"))
    np.savez_compressed(os.path.join(OUT, f"{NAME}_dev_logits.npz"), **{f"d:{k}": v.astype(np.float16) for k, v in dlg.items()})

if CFG.get("dump_teacher_logits"):
    every = dict(SPLITS.get("train", {})); every.update(SPLITS.get("dev", {}))
    tl = predict_logits(every)
    np.savez_compressed(os.path.join(OUT, f"{NAME}_teacher_logits.npz"),
                        __meta__=json.dumps(dict(labels=LABELS, run=NAME)),
                        **{f"d:{k}": v.astype(np.float16) for k, v in tl.items()})
    log(f"teacher logits written for {len(tl)} docs")

if TEST:
    t1 = time.time()
    tlg = predict_logits(TEST)
    np.savez_compressed(os.path.join(OUT, f"{NAME}_test_logits.npz"), **{f"d:{k}": v.astype(np.float16) for k, v in tlg.items()})
    pk = labels_from(tlg)
    ids = [TEST[k]["id"] for k in pk]
    clash = {i for i in ids if ids.count(i) > 1}
    assert not clash, f"doc ids repeat across domains: {sorted(clash)[:3]} — cannot merge into one prediction.json"
    pred = {TEST[k]["id"]: v for k, v in pk.items()}
    n_sent = rd.validate_prediction(pred, {TEST[k]["id"]: TEST[k] for k in TEST}, LABELS)
    pj = os.path.join(OUT, f"{NAME}_prediction.json")
    json.dump(pred, open(pj, "w", encoding="utf-8"), ensure_ascii=False)
    zp = os.path.join(OUT, f"{NAME}_submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pj, arcname="prediction.json")
    from collections import Counter
    result.update(test_sentences=n_sent, test_infer_seconds=time.time() - t1)
    log(f"test: {len(pred)} docs, {n_sent} sentences | predicted labels {dict(Counter(l for v in pred.values() for l in v).most_common())}")
    log(f"wrote {zp}")
json.dump(result, open(os.path.join(OUT, f"{NAME}_result.json"), "w"), indent=1)
log(f"done: {NAME} | best dev {SELECT} macro-F1 {best:.4f} | train {train_min:.1f} min | peak {peak_gb:.1f} GB | "
    f"trainable {n_train/1e6:.3f}M of {n_total/1e6:.2f}M")

In [ ]:
%%writefile phase1_loader.py
"""phase1_loader.py — load a Phase-1 model with its trained weights."""
import importlib, json, os

import torch

REGISTRY = os.environ.get("PHASE1_REGISTRY", "phase1_registry.json")


def load_phase1(name, map_location="cpu"):
    reg = json.load(open(REGISTRY))
    assert name in reg, f"{name!r} not in {REGISTRY}: {sorted(reg)}"
    r = reg[name]
    mod = importlib.import_module(r["module"])
    cfg = dict(r["model_cfg"])
    model = mod.build_model(cfg)
    ck = torch.load(r["ckpt"], map_location=map_location, weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, f"{name}: missing {missing[:5]} unexpected {unexpected[:5]}"
    val = ck.get("val_loss") if isinstance(ck, dict) else None
    return model, dict(model_cfg=cfg, ckpt=r["ckpt"], module=r["module"], val_loss=r.get("val_loss", val))

In [ ]:
%%writefile rr_dummy.py
"""rr_dummy.py — PIPELINE TEST ONLY (smoke test): mean of token embeddings per sentence slot. Never submit it."""
import torch
import torch.nn as nn


class _Dummy(nn.Module):
    def __init__(self, vocab, n):
        super().__init__()
        self.emb = nn.Embedding(vocab, 64)
        self.out = nn.Linear(64, n)

    def forward(self, input_ids, attention_mask, sent_ids, n_slots):
        h = self.emb(input_ids)
        onehot = (sent_ids[..., None] == torch.arange(n_slots, device=input_ids.device)).to(h.dtype)
        pooled = onehot.transpose(1, 2) @ h / onehot.sum(1)[..., None].clamp(min=1)
        return self.out(pooled)


def build_rr_model(model_cfg, num_labels):
    return _Dummy(int(model_cfg.get("vocab_size", 32000)), num_labels)

In [ ]:
%%writefile p1_biscale.py
"""model.py for the Bi-Scale generation notebook."""
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================== copied from the Bi-Scale training notebook ==============================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, seq_len):
        if seq_len > self.cos_cached.shape[2]:
            self._build_cache(seq_len)
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary(x, cos, sin):
    return x * cos + rotate_half(x) * sin

class BiScaleGQA(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout, ctx_len, rope_theta, is_local=False, window_size=256):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups   = n_heads // n_kv_heads
        self.d_head     = d_model // n_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.d_head, max_seq=ctx_len, theta=rope_theta)

        causal = torch.tril(torch.ones(ctx_len, ctx_len))
        if is_local:
            causal = causal * torch.triu(torch.ones(ctx_len, ctx_len), diagonal=-window_size)
        self.register_buffer("mask", causal.view(1, 1, ctx_len, ctx_len))


        for proj in [self.q_proj, self.k_proj, self.v_proj, self.o_proj]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T, C = x.shape
        dh = self.d_head

        q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)

        cos, sin = self.rope(T)
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        k = k.repeat_interleave(self.n_groups, dim=1)
        v = v.repeat_interleave(self.n_groups, dim=1)

        scale = dh ** -0.5
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))

        out = (att @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        for proj in [self.gate, self.up, self.down]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class LearnedGatedResidual(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Linear(dim, dim, bias=True)

        nn.init.normal_(self.gate.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.gate.bias, 2.0)

    def forward(self, x, sublayer):
        g = torch.sigmoid(self.gate(x))
        return x + g * sublayer

class BiScaleLegalBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        is_local = (layer_idx < (cfg["n_layers"] // 2))
        self.norm1 = RMSNorm(cfg["d_model"])
        self.attn  = BiScaleGQA(
            cfg["d_model"], cfg["n_heads"], cfg["n_kv_heads"],
            cfg["dropout"], cfg["ctx_len"], cfg["rope_theta"],
            is_local=is_local, window_size=cfg["local_window"]
        )
        self.res1 = LearnedGatedResidual(cfg["d_model"])
        self.norm2 = RMSNorm(cfg["d_model"])
        self.ffn   = SwiGLU(cfg["d_model"], cfg["d_ff"])
        self.res2  = LearnedGatedResidual(cfg["d_model"])
        self.drop  = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        x = self.res1(x, self.drop(self.attn(self.norm1(x))))
        x = self.res2(x, self.drop(self.ffn(self.norm2(x))))
        return x

class BiScaleLegalLLaMA(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.ctx     = cfg["ctx_len"]
        self.tok_emb = nn.Embedding(vocab_size, cfg["d_model"])
        self.drop    = nn.Dropout(cfg["dropout"])
        self.blocks  = nn.ModuleList([BiScaleLegalBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.norm    = RMSNorm(cfg["d_model"])
        self.lm_head = nn.Linear(cfg["d_model"], vocab_size, bias=False)


        self.lm_head.weight = self.tok_emb.weight


        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
# ======================================= end of copied code =======================================


_ORIGINAL_ATTN_FORWARD = BiScaleGQA.forward


def _sdpa_attn_forward(self, x):
    B, T, C = x.shape
    dh = self.d_head
    q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    cos, sin = self.rope(T)
    q = apply_rotary(q, cos.to(q.dtype), sin.to(q.dtype))
    k = apply_rotary(k, cos.to(k.dtype), sin.to(k.dtype))
    k = k.repeat_interleave(self.n_groups, dim=1)
    v = v.repeat_interleave(self.n_groups, dim=1)
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=self.mask[:, :, :T, :T].bool(),
                                       dropout_p=0.0, scale=dh ** -0.5)
    return self.o_proj(y.transpose(1, 2).contiguous().view(B, T, -1))


def use_sdpa(flag=True):
    BiScaleGQA.forward = _sdpa_attn_forward if flag else _ORIGINAL_ATTN_FORWARD


class BiScaleForHarness(BiScaleLegalLLaMA):
    def forward(self, input_ids, prefix_lens=None):
        return super().forward(input_ids)[0]


def build_model(model_cfg):
    use_sdpa(model_cfg.get("use_sdpa", True))
    return BiScaleForHarness(model_cfg, model_cfg["vocab_size"])

In [ ]:
# 3 — data check: formats, sizes, label distribution per domain, sentence lengths
!python test_rr_data.py | tail -1
import rr_data as rd, importlib; importlib.reload(rd)
from tokenizers import ByteLevelBPETokenizer
_tok = ByteLevelBPETokenizer(vocab="tokenizer/vocab.json", merges="tokenizer/merges.txt")
enc = lambda s: _tok.encode(s).ids
LABELS = rd.read_label_vocab(P3["label_vocab.json"])
print("labels:", LABELS)
for f in ["CL_train.json", "CL_test.json"]:
    raw = json.load(open(P3[f]))
    ex = raw[0] if isinstance(raw, list) else next(iter(raw.items()))
    print(f"{f} layout: {type(raw).__name__}; example: {str(ex)[:250]}")
ALL = {}
for dom in ("CL", "IT"):
    for split in ("train", "dev", "test"):
        docs = rd.load_docs(P3[f"{dom}_{split}.json"])
        ALL[(dom, split)] = docs
        st = rd.dataset_stats(docs, rd.tokenize_sentences(docs, enc), LABELS)
        print(f"\n{dom} {split}: " + json.dumps(st))
ids = [d for (dom, s), v in ALL.items() if s == "test" for d in v]
print("\ntest doc ids unique across CL+IT:", len(ids) == len(set(ids)))

In [ ]:
# 4 — Phase-1 models -> phase1_registry.json (same as Phase 2)
REG = {}
V = _tok.get_vocab_size()
def near(ckpt, fn):
    root = os.path.dirname(os.path.dirname(ckpt)) if os.path.basename(os.path.dirname(ckpt)) == "run" else os.path.dirname(ckpt)
    hits = sorted(glob.glob(f"{root}/**/{fn}", recursive=True), key=len)
    return hits[0] if hits else None
for name in ["prefixlm", "encdec", "arch4"]:
    ck = find(f"{name}_best.pt")
    if not ck:
        print(f"{name}: NOT FOUND (attach phase1-models)"); continue
    shutil.copy(near(ck[0], "model.py"), f"p1_{name}.py")
    cfgj = json.load(open(near(ck[0], "run_config.json")))
    mc = dict(cfgj["model"]); mc.setdefault("vocab_size", V); mc.setdefault("max_seq_len", int(cfgj["train"]["ctx_len"]))
    REG[name] = dict(module=f"p1_{name}", model_cfg=mc, ckpt=ck[0])
bs = find("stage2_done.pt")
if bs:
    REG["biscale"] = dict(module="p1_biscale", ckpt=bs[0],
                          model_cfg=dict(n_layers=16, n_heads=12, n_kv_heads=4, d_model=768, d_ff=2048, ctx_len=1024,
                                         local_window=256, dropout=0.1, rope_theta=10000.0, vocab_size=V))
json.dump(REG, open("phase1_registry.json", "w"), indent=1)
from phase1_loader import load_phase1
for name in REG:
    m, info = load_phase1(name)
    print(f"{name:9s} {sum(p.numel() for p in m.parameters())/1e6:7.2f}M params | {info['ckpt']}")
    del m

In [ ]:
# 5 — experiment config helper + parallel runner
TODO = "TODO"
R = "/kaggle/working/p3runs"
def make_cfg(run_name, model, train, data=None, kd=None, dump_teacher=False, module="rr_model", select="combined"):
    d = dict(tokenizer_dir="tokenizer", label_vocab=P3["label_vocab.json"],
             domains={dom: dict(train=P3[f"{dom}_train.json"], dev=P3[f"{dom}_dev.json"], test=P3[f"{dom}_test.json"])
                      for dom in ("CL", "IT")},
             train_splits=["train"], train_domains=["CL", "IT"],
             mode=TODO,
             window=TODO,
             sep=TODO,
             target_frac=0.5,
             prefix_space=True, max_sent_tokens=None)
    d.update(data or {})
    cfg = dict(run_name=run_name, out_dir=f"{R}/{run_name}", model_module=module, data=d, model=model, train=train,
               kd=kd, dump_teacher_logits=dump_teacher, select=select)
    todo = [k for k, v in {**d, **train, **(kd or {}), **model}.items() if v == TODO]
    assert not todo, f"{run_name}: fill {todo}"
    os.makedirs(cfg["out_dir"], exist_ok=True)
    p = f"{cfg['out_dir']}/config.json"; json.dump(cfg, open(p, "w"), indent=1)
    return p

def run_parallel(cfg_paths, timeout_min=180):
    procs, queue = [], list(cfg_paths)
    while queue or procs:
        while queue and len(procs) < max(1, NGPU):
            cp = queue.pop(0); gpu = [g for g in range(max(1, NGPU)) if g not in [x[2] for x in procs]][0]
            lf = open(os.path.join(os.path.dirname(cp), "train.log"), "w")
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
            procs.append((subprocess.Popen([sys.executable, "rr_train.py", cp], stdout=lf, stderr=subprocess.STDOUT, env=env),
                          cp, gpu, lf, time.time()))
            print(f"started {cp} on GPU {gpu}")
        time.sleep(15)
        for pr in list(procs):
            p, cp, gpu, lf, t0 = pr
            if p.poll() is not None or time.time() - t0 > timeout_min * 60:
                if p.poll() is None: p.kill()
                lf.close(); procs.remove(pr)
                tail = open(os.path.join(os.path.dirname(cp), "train.log")).read().splitlines()[-40:]
                print(f"\n==== {cp} exit {p.returncode} ({(time.time()-t0)/60:.1f} min)\n" + "\n".join(tail))

def summary(names):
    for n in names:
        z, res = f"{R}/{n}/{n}_submission.zip", f"{R}/{n}/{n}_result.json"
        if os.path.exists(z) and os.path.exists(res):
            r = json.load(open(res)); dv = r.get("dev", {})
            f = lambda k: dv.get(k, {}).get("macro_f1", float("nan"))
            print(f"OK  {n:20s} dev macro-F1 combined {f('combined'):.4f} | CL {f('CL'):.4f} | IT {f('IT'):.4f} | "
                  f"trainable {r['params_trainable']/1e6:.2f}M of {r['params_total']/1e6:.1f}M | {r['train_minutes']:.1f} min -> {z}")
        else:
            print(f"FAILED {n}: read {R}/{n}/train.log")

In [ ]:
# 6 — smoke test: dummy model, sentence + context mode, teacher dump -> KD student, both-domain zip
SM_T = dict(epochs=1, lr=1e-3, batch_size=16, grad_accum=1, warmup_frac=0.0, weight_decay=0.0, max_minutes=3)
a = make_cfg("smoke_sent", dict(vocab_size=V), SM_T, dict(mode="sentence", window=256, sep=None), module="rr_dummy",
             dump_teacher=True)
b = make_cfg("smoke_ctx", dict(vocab_size=V), SM_T, dict(mode="context", window=512, sep="</s>"), module="rr_dummy")
run_parallel([a, b])
c = make_cfg("smoke_kd", dict(vocab_size=V), SM_T, dict(mode="context", window=512, sep="</s>"), module="rr_dummy",
             kd=dict(teacher_logits=f"{R}/smoke_sent/smoke_sent_teacher_logits.npz", temperature=2.0, alpha=0.5))
run_parallel([c])
summary(["smoke_sent", "smoke_ctx", "smoke_kd"])
assert all(os.path.exists(f"{R}/{n}/{n}_submission.zip") for n in ["smoke_sent", "smoke_ctx", "smoke_kd"]), "SMOKE TEST FAILED"
print("SMOKE TEST PASSED (pipeline only: the dummy model's scores mean nothing)")

In [ ]:
%%writefile rr_model.py
"""rr_model.py — Phase-3 model."""
import torch
import torch.nn as nn

from phase1_loader import load_phase1


def build_rr_model(model_cfg: dict, num_labels: int) -> nn.Module:
    raise NotImplementedError("write your Phase-3 model here")

In [ ]:
# 8 — model check: output shape [B, S, C], trainable params, gradients reach every trainable tensor, memory
import importlib, rr_model; importlib.reload(rr_model)
CHECK_MODEL = dict(TODO="TODO")
C = len(LABELS)
m = rr_model.build_rr_model(dict(CHECK_MODEL), C).cuda().train()
B, T, S = 4, 512, 12
x = torch.randint(5, V, (B, T), device="cuda"); msk = torch.ones_like(x, dtype=torch.bool)
sid = (torch.arange(T, device="cuda") * S // T)[None].expand(B, -1).clone()
torch.cuda.reset_peak_memory_stats()
with torch.autocast("cuda", dtype=torch.float16):
    out = m(x, msk, sid, S)
assert tuple(out.shape) == (B, S, C), f"logits must be [B, n_slots, num_labels], got {tuple(out.shape)}"
out.float().mean().backward()
nog = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None]
print(f"total {sum(p.numel() for p in m.parameters())/1e6:.2f}M | trainable {sum(p.numel() for p in m.parameters() if p.requires_grad)/1e6:.3f}M"
      f" | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB | trainable but no gradient: {nog[:10]}")
del m, out; torch.cuda.empty_cache()

In [ ]:
# 9 — EXPERIMENTS (YOURS): make_cfg(...) per experiment, run_parallel([...]) two at a time, then summary([...])

In [ ]:
# 10 — results table (report) + download links
import pandas as pd
from IPython.display import FileLink, display
rows = []
for rp in sorted(glob.glob(f"{R}/*/*_result.json")):
    r = json.load(open(rp))
    if r["run"].startswith("smoke"): continue
    dv = r.get("dev", {})
    rows.append(dict(run=r["run"], dev_combined=dv.get("combined", {}).get("macro_f1"), dev_CL=dv.get("CL", {}).get("macro_f1"),
                     dev_IT=dv.get("IT", {}).get("macro_f1"), trainable_M=r["params_trainable"]/1e6, total_M=r["params_total"]/1e6,
                     train_min=r["train_minutes"], peak_GB=r["peak_mem_gb"], mode=r["mode"], window=r["window"]))
df = pd.DataFrame(rows).sort_values("dev_combined", ascending=False); display(df)
df.to_csv(f"{R}/results.csv", index=False)
for n in df.run: display(FileLink(f"p3runs/{n}/{n}_submission.zip"))